# GridSearchCV – Ajuste de hiperparámetros – UNCAus
**Doctorado en Informática – UTN FRRe/UNNE/UNAM**  
Curso: Introducción al Aprendizaje de Máquina  
Integrantes: Stella Gerzel · Juan Manuel Goya · Gustavo Ebel

---
Este notebook ajusta los hiperparámetros de Random Forest y XGBoost
usando GridSearchCV sobre los 3 problemas de clasificación.

**Tiempo estimado de ejecución: 5-10 minutos** (180 entrenamientos por problema)

In [ ]:
# CELDA 1 — Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
print('Librerías OK')

In [ ]:
# CELDA 2 — Cargar DataSet_modelado_base
# Input: DataSet_modelado_base_1_.xlsx (output del notebook 03 de Stella)
# Ya tiene: columnas descartadas, faltantes normalizados, nota conservada
# NO tiene: imputación, codificación, split — eso se hace acá

from google.colab import files
uploaded = files.upload()

ruta = next(iter(uploaded))
df = pd.read_excel(ruta, sheet_name='Datos_modelado_base')
df_w = df.copy()

# Separar predictoras del target (ya no hay que descartar nada)
X_base = df_w.drop(columns=['nota']).copy()

# 3 targets binarios
y1 = df_w['nota'].map({'A':0,'R':1,'U':1})
mask_p2 = df_w['nota'].isin(['A','R'])
X_p2 = X_base[mask_p2].reset_index(drop=True)
y2   = df_w.loc[mask_p2,'nota'].map({'A':0,'R':1}).reset_index(drop=True)
mask_p3 = df_w['nota'].isin(['A','U'])
X_p3 = X_base[mask_p3].reset_index(drop=True)
y3   = df_w.loc[mask_p3,'nota'].map({'A':0,'U':1}).reset_index(drop=True)

# Splits estratificados 75/25
X1_tr,X1_te,y1_tr,y1_te = train_test_split(X_base,y1,test_size=0.25,stratify=y1,random_state=42)
X2_tr,X2_te,y2_tr,y2_te = train_test_split(X_p2,  y2,test_size=0.25,stratify=y2,random_state=42)
X3_tr,X3_te,y3_tr,y3_te = train_test_split(X_p3,  y3,test_size=0.25,stratify=y3,random_state=42)

cols_cat = X_base.select_dtypes('object').columns.tolist()
cols_num = X_base.select_dtypes('number').columns.tolist()

# Preprocesador
def convertir_a_texto(X):
    """Convierte columnas a texto para evitar errores con tipos mixtos."""
    return X.astype(str)

preprocesador = ColumnTransformer([
    ('cat', Pipeline([
        ('imp',       SimpleImputer(strategy='constant', fill_value='Sin dato')),
        ('to_string', FunctionTransformer(convertir_a_texto, validate=False)),
        ('enc',       OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cols_cat),
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('esc', StandardScaler())
    ]), cols_num)
])

print(f'Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Variables predictoras: {X_base.shape[1]} ({len(cols_cat)} categóricas, {len(cols_num)} numéricas)')
print(f'Train P1:{len(X1_tr)} | P2:{len(X2_tr)} | P3:{len(X3_tr)}')

---
## GridSearchCV — Regresión Logística (agregado)

In [ ]:
# CELDA 3 — GridSearchCV para Regresión Logística (línea base)
#
# AGREGADO: el informe (sección 9.4) documenta la optimización de la
# Regresión Logística con GridSearchCV, pero este notebook no la incluía.
# Se agrega con la misma grilla declarada en el informe. Con el escalado
# ya corregido en el preprocesador, lbfgs converge sin problemas.

grilla_lr = {
    'modelo__C':            [0.01, 0.1, 1, 10],
    'modelo__penalty':      ['l2'],
    'modelo__solver':       ['lbfgs'],
    'modelo__class_weight': [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

problemas = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

mejores_lr = {}
print('Buscando mejores hiperparámetros para Regresión Logística...\n')

for nombre, X_tr, X_te, y_tr, y_te in problemas:
    pipe = Pipeline([
        ('prep',   preprocesador),
        ('modelo', LogisticRegression(max_iter=1000, random_state=42))
    ])
    gs = GridSearchCV(pipe, grilla_lr, cv=cv,
                      scoring='roc_auc', n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr)

    mejores_lr[nombre] = gs
    auc_test = roc_auc_score(y_te, gs.predict_proba(X_te)[:,1])

    print(f'── {nombre} ──')
    print(f'  Mejor combinación:      {gs.best_params_}')
    print(f'  ROC-AUC en CV (train):  {gs.best_score_:.3f}')
    print(f'  ROC-AUC en test:        {auc_test:.3f}')
    print()

---
## GridSearchCV — Random Forest

In [ ]:
# CELDA 3 — GridSearchCV para Random Forest
#
# La grilla define qué valores probar para cada hiperparámetro
# GridSearchCV prueba todas las combinaciones (3×4×3 = 36) con CV 5-fold
# y se queda con la que da mayor ROC-AUC promedio
#
# IMPORTANTE: los nombres de los parámetros llevan el prefijo 'modelo__'
# porque están dentro del pipeline (prep → modelo)

# --- Versión anterior (grilla que no coincidía con el informe, comentada) ---
# grilla_rf = {
#     'modelo__n_estimators':    [50, 100, 200],
#     'modelo__max_depth':       [5, 10, 20, None],
#     'modelo__min_samples_leaf': [1, 2, 5]
# }
# CORRECCIÓN: grilla alineada con el informe (sección 9.5). Prueba
# 2×3×2×2×2 = 48 combinaciones con CV 5-fold (240 fits por problema).
grilla_rf = {
    'modelo__n_estimators':      [100, 300],
    'modelo__max_depth':         [None, 5, 10],
    'modelo__min_samples_split': [2, 5],
    'modelo__min_samples_leaf':  [1, 3],
    'modelo__class_weight':      [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

problemas = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

mejores_rf = {}
print('Buscando mejores hiperparámetros para Random Forest...')
print('(puede tardar 5-10 minutos)\n')

for nombre, X_tr, X_te, y_tr, y_te in problemas:
    pipe = Pipeline([
        ('prep',   preprocesador),
        # ('modelo', RandomForestClassifier(class_weight='balanced', random_state=42))
        # CORRECCIÓN: class_weight ahora se busca en la grilla, no se fija
        ('modelo', RandomForestClassifier(random_state=42))
    ])
    gs = GridSearchCV(pipe, grilla_rf, cv=cv,
                      scoring='roc_auc', n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr)

    mejores_rf[nombre] = gs
    auc_test = roc_auc_score(y_te, gs.predict_proba(X_te)[:,1])

    print(f'── {nombre} ──')
    print(f'  Mejor combinación:      {gs.best_params_}')
    print(f'  ROC-AUC en CV (train):  {gs.best_score_:.3f}')
    print(f'  ROC-AUC en test:        {auc_test:.3f}')
    print()

---
## GridSearchCV — XGBoost

In [ ]:
# CELDA 4 — GridSearchCV para XGBoost
#
# XGBoost no tiene class_weight — usa scale_pos_weight para el desbalance
# scale_pos_weight = n_negativos / n_positivos  (calculado por problema)
#
# Hiperparámetros propios de XGBoost:
# n_estimators: cantidad de árboles (igual que RF)
# max_depth:    profundidad máxima (igual que RF)
# learning_rate: qué tan rápido aprende cada árbol nuevo
#   (bajo = aprende despacio pero más preciso, alto = rápido pero puede sobreajustar)

# --- Versión anterior (grilla que no coincidía con el informe, comentada) ---
# grilla_xgb = {
#     'modelo__n_estimators':  [50, 100, 200],
#     'modelo__max_depth':     [3, 5, 7],
#     'modelo__learning_rate': [0.01, 0.1, 0.3]
# }

mejores_xgb = {}
print('Buscando mejores hiperparámetros para XGBoost...')
print('(puede tardar 5-10 minutos)\n')

for nombre, X_tr, X_te, y_tr, y_te in problemas:
    # Calcular scale_pos_weight para este problema (informe, sección 9.6)
    n_neg = (y_tr == 0).sum()
    n_pos = (y_tr == 1).sum()
    spw   = n_neg / n_pos

    # CORRECCIÓN: grilla alineada con el informe. scale_pos_weight ahora
    # se busca en la grilla [1, balanceado] en vez de fijarse en el
    # constructor, y se arma DENTRO del bucle porque spw cambia por problema.
    grilla_xgb = {
        'modelo__n_estimators':     [100, 300],
        'modelo__learning_rate':    [0.05, 0.1],
        'modelo__max_depth':        [3, 5],
        'modelo__subsample':        [0.8, 1.0],
        'modelo__colsample_bytree': [0.8, 1.0],
        'modelo__scale_pos_weight': [1, round(spw, 4)],
    }

    pipe = Pipeline([
        ('prep',   preprocesador),
        # ('modelo', XGBClassifier(scale_pos_weight=spw, ...))  # <- ahora se busca en la grilla
        ('modelo', XGBClassifier(eval_metric='logloss',
                                 random_state=42, verbosity=0))
    ])
    gs = GridSearchCV(pipe, grilla_xgb, cv=cv,
                      scoring='roc_auc', n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr)

    mejores_xgb[nombre] = gs
    auc_test = roc_auc_score(y_te, gs.predict_proba(X_te)[:,1])

    print(f'── {nombre} ──')
    print(f'  Mejor combinación:      {gs.best_params_}')
    print(f'  ROC-AUC en CV (train):  {gs.best_score_:.3f}')
    print(f'  ROC-AUC en test:        {auc_test:.3f}')
    print()

---
## Comparación final: modelos base vs. modelos ajustados

In [ ]:
# CELDA 5 — Tabla comparativa
#
# ROC-AUC base = resultados del notebook anterior (parámetros por defecto)
# ROC-AUC ajustado = resultados de GridSearchCV

base_rf = {'P1 Riesgo global': 0.596,
           'P2 Desaprobacion': 0.625,
           'P3 Abandono':      0.789}

# NOTA: base_rf está copiado a mano de una corrida anterior de
# Modelado_UNCAus.ipynb. Si se re-ejecuta Modelado con el pipeline
# corregido (mediana+escalado / 'Sin dato'), estos valores cambian y
# hay que actualizarlos.
filas = []
for nombre, X_tr, X_te, y_tr, y_te in problemas:
    auc_lr  = roc_auc_score(y_te, mejores_lr[nombre].predict_proba(X_te)[:,1])
    auc_rf  = roc_auc_score(y_te, mejores_rf[nombre].predict_proba(X_te)[:,1])
    auc_xgb = roc_auc_score(y_te, mejores_xgb[nombre].predict_proba(X_te)[:,1])
    filas.append({
        'Problema':          nombre,
        'RF base':           base_rf[nombre],
        'LogReg ajustado':   round(auc_lr, 3),
        'RF ajustado':       round(auc_rf, 3),
        'XGB ajustado':      round(auc_xgb, 3),
        'Mejora RF':         round(auc_rf - base_rf[nombre], 3),
    })

tabla = pd.DataFrame(filas).set_index('Problema')
print('ROC-AUC en test — comparación:')
print(tabla.to_string())
print()
print('Mejora RF = diferencia entre RF ajustado y RF base')
print('Valores positivos → GridSearchCV mejoró el modelo')

In [ ]:
# CELDA 6 — Matrices de confusión de los mejores modelos ajustados

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
titulos   = ['P1 Riesgo global', 'P2 Desaprobacion', 'P3 Abandono']

for col, (nombre, X_tr, X_te, y_tr, y_te) in enumerate(problemas):
    # Fila 0: mejor RF ajustado
    ConfusionMatrixDisplay.from_estimator(
        mejores_rf[nombre], X_te, y_te,
        display_labels=['Aprobó','Riesgo'],
        colorbar=False, ax=axes[0, col])
    axes[0, col].set_title(f'{nombre}\nRF ajustado', fontweight='bold')

    # Fila 1: mejor XGB ajustado
    ConfusionMatrixDisplay.from_estimator(
        mejores_xgb[nombre], X_te, y_te,
        display_labels=['Aprobó','Riesgo'],
        colorbar=False, ax=axes[1, col])
    axes[1, col].set_title(f'{nombre}\nXGB ajustado', fontweight='bold')

plt.suptitle('Matrices de confusión — modelos ajustados con GridSearchCV',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# CELDA 7 — Mejores hiperparámetros encontrados (resumen para el informe)
print('=' * 60)
print('MEJORES HIPERPARÁMETROS — PARA EL INFORME')
print('=' * 60)
for nombre, _, _, _, _ in problemas:
    params_lr  = mejores_lr[nombre].best_params_
    params_rf  = mejores_rf[nombre].best_params_
    params_xgb = mejores_xgb[nombre].best_params_
    print(f'\n{nombre}:')
    print(f'  LR:  C={params_lr["modelo__C"]}, '
          f'penalty={params_lr["modelo__penalty"]}, '
          f'solver={params_lr["modelo__solver"]}, '
          f'class_weight={params_lr["modelo__class_weight"]}')
    # CORRECCIÓN: min_samples_split y class_weight también forman parte
    # de la grilla ahora, se agregan al resumen.
    print(f'  RF:  n_estimators={params_rf["modelo__n_estimators"]}, '
          f'max_depth={params_rf["modelo__max_depth"]}, '
          f'min_samples_split={params_rf["modelo__min_samples_split"]}, '
          f'min_samples_leaf={params_rf["modelo__min_samples_leaf"]}, '
          f'class_weight={params_rf["modelo__class_weight"]}')
    print(f'  XGB: n_estimators={params_xgb["modelo__n_estimators"]}, '
          f'learning_rate={params_xgb["modelo__learning_rate"]}, '
          f'max_depth={params_xgb["modelo__max_depth"]}, '
          f'subsample={params_xgb["modelo__subsample"]}, '
          f'colsample_bytree={params_xgb["modelo__colsample_bytree"]}, '
          f'scale_pos_weight={params_xgb["modelo__scale_pos_weight"]}')
print()
print('Siguiente paso: importancia de variables con los mejores modelos.')

In [ ]:
# CELDA 8 — Métricas con umbral 0.50 (tablas de las secciones 9.4 a 9.6 del informe)
#
# AGREGADO: el informe reporta CV ROC-AUC, Test ROC-AUC, Accuracy,
# Precision, Recall y F1 con umbral 0.50 para cada modelo y problema,
# pero el notebook no las calculaba.

modelos_gs = {
    'Regresión Logística': mejores_lr,
    'Random Forest':       mejores_rf,
    'XGBoost':              mejores_xgb,
}

for nombre, X_tr, X_te, y_tr, y_te in problemas:
    filas = []
    for etiqueta, dic in modelos_gs.items():
        gs     = dic[nombre]
        y_prob = gs.predict_proba(X_te)[:,1]
        y_pred = (y_prob >= 0.50).astype(int)
        filas.append({
            'Modelo':      etiqueta,
            'CV ROC-AUC':  round(gs.best_score_, 4),
            'Test ROC-AUC':round(roc_auc_score(y_te, y_prob), 4),
            'Accuracy':    round(accuracy_score(y_te, y_pred), 4),
            'Precision':   round(precision_score(y_te, y_pred, zero_division=0), 4),
            'Recall':      round(recall_score(y_te, y_pred), 4),
            'F1':          round(f1_score(y_te, y_pred, zero_division=0), 4),
        })
    print(f'── {nombre} — métricas con umbral 0.50 ──')
    print(pd.DataFrame(filas).set_index('Modelo').to_string())
    print()